# TRACK C — E-MULTIDEV, crossed 5 devices × 2 seeds

**C0** Kodak_M1063 discovery, splits and fingerprints (~1 h) · **C1** 10 adapters, generation,
measurement, mixed model (~8 h).

### Why crossed
One adapter per device confounds **device identity with training seed**. 5 × 2 = 10 adapters
supports θ_{d,s} = μ + u_d + v_{d,s}, separating between-device variation from adapter-training
variation. Two seeds per device will not resolve a device × seed interaction precisely, but it
prevents complete confounding.

### What five bodies buy that the D200 pair could not
**κ_model by leave-one-device-out:** κ̂^(−d) = ¼ Σ_{j≠d} K̂_j, residualising device d against the
*other four*. Report every result **both raw and κ-residualised**.

### Protocol discipline
Primary training protocol is **fixed** — identical optimiser, schedule, steps, effective batch, no
norm-dependent intervention. Adaptation-strength matching is **secondary and pre-specified**:
‖lora_B‖ as a covariate, *or* inference-time adapter scaling at matched effective norm, *or* one
additional norm-targeted arm. Norm-targeted early stopping changes the update count and substitutes
one confound for another.

Set `HF_TOKEN` in Colab Secrets, A100 runtime, **Run all**.

In [ ]:
# %% C-CFG
import os, sys, json, glob, re, math, time, gc, shutil, hashlib, subprocess, random, itertools
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
import csv as _csv
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"]=(userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
DRIVE="/content/drive/MyDrive"
if not os.path.isdir(DRIVE): raise SystemExit("Drive not mounted")

@dataclass
class CFG:
    STAGES: tuple = ("C0","C1","C2","C3","C4")
    DRESDEN: str = "forensic_datasets/dresden/Dresden_Exp"
    OUT: str = "inv_channel/E_MULTIDEV"
    MODEL: str = "stabilityai/stable-diffusion-3.5-medium"
    CAM_MODEL: str = "Kodak_M1063"
    N_DEV: int = 5
    CROP: int = 1024
    N_E1: int = 80; N_E2: int = 140; N_T: int = 50; N_H: int = 40; GUARD: int = 10
    SEEDS: tuple = (0,1)                      # 2 seeds per device -> 10 adapters
    RANK: int = 16
    TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    LR: float = 1e-4; STEPS: int = 2000; BATCH: int = 2; GRAD_ACC: int = 2
    CAPTION: str = "a photograph, sks style"
    G_PER: int = 500; GEN_SEED_BASE: int = 770000
    GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 4
    N_PERM: int = 10000; ALPHA: float = 0.01
    GRAD_CKPT: bool = False
C=CFG(); MEAS=C.CROP
CFG_SHA=hashlib.sha256(json.dumps({k:v for k,v in asdict(C).items() if k!="STAGES"},
                                  sort_keys=True,default=str).encode()).hexdigest()
ROOT=os.path.join(DRIVE,C.OUT)
for d in ["","manifest","fingerprints","train_png","adapters","gens","csv","meta","figs"]:
    os.makedirs(os.path.join(ROOT,d),exist_ok=True)
CACHE="/content/md_cache"; os.makedirs(CACHE,exist_ok=True)
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False

subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","peft","accelerate","safetensors","sentencepiece","protobuf"],check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy import stats as sps
try:
    import peft.import_utils as _piu; _piu.is_torchao_available=lambda: False
    import peft.tuners.lora.torchao as _plt; _plt.is_torchao_available=lambda: False
    print("[ENV] peft torchao dispatcher disabled")
except Exception: pass
Image.MAX_IMAGE_PIXELS=None
DEV="cuda" if torch.cuda.is_available() else "cpu"
DT=torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
np.seterr(divide="raise",invalid="raise"); _MD=DEV
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])
if DT==torch.float16: raise SystemExit("fp16 fallback — use an A100/L4/H100 (bf16 required)")
def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")
def append_row(p,h,r):
    new=not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def done_keys(p,cols):
    if not os.path.exists(p): return set()
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _csv.DictReader(f)}
def ajson(o,p,**kw):
    t=p+".tmp"; json.dump(o,open(t,"w"),**kw); os.replace(t,p)
def DEVS(): return [f"D{i}" for i in range(C.N_DEV)]
def all_arms(): return [(f"{d}_s{s}", d, s) for s in C.SEEDS for d in DEVS()]
def all_tags(): return [t for t,_,_ in all_arms()]
print(f"protocol SHA {CFG_SHA[:16]} | {len(all_arms())} adapters | device={DEV} dtype={DT}")

In [ ]:
# %% forensic core — byte-identical settings to the base study
def _cp(fp): return os.path.join(CACHE,hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16]+".npy")
def load_lum(fp,cache=True):
    if cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        if min(W,H)<MEAS: HALT(f"{fp}: {W}x{H} < {MEAS}")
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    l=(0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
    if cache: np.save(_cp(fp),l)
    return l
def load_rgb(fp):
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        return np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
def _c2(x,k):
    xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
    if _MD=="cuda": return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()
def wres(img):
    s0=np.float32(4.0)
    co=pywt.wavedec2(img,"db8",level=4,mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch=ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v=np.maximum(_c2(ch*ch,np.ones((w,w),np.float32)/(w*w))-s0,0.0)
                vm=v if vm is None else np.minimum(vm,v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n=pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
    r=n-n.mean(); sd=float(r.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)
def zm(K):
    K=K-K.mean(axis=1,keepdims=True); return K-K.mean(axis=0,keepdims=True)
def wdft(K):
    F=np.fft.fft2(K.astype(np.float32)); mag=(np.abs(F)/K.shape[0]).astype(np.float32)
    sig=max(float(np.median(mag))/0.6745,1e-12); s2=np.float32(sig**2); vm=None
    for w in (3,5,7,9):
        k=np.ones((w,w),np.float32)/(w*w); mu=_c2(mag,k)
        v=np.maximum(_c2(mag*mag,k)-mu*mu,0.0); vm=v if vm is None else np.minimum(vm,v)
    return np.real(np.fft.ifft2(F*(s2/(vm+s2)).astype(np.float32))).astype(np.float32)
def estK(files):
    num=den=None
    for fp in files:
        Y=load_lum(fp); W=wres(Y)
        num=W*Y if num is None else num+W*Y; den=Y*Y if den is None else den+Y*Y
    return wdft(zm((num/np.maximum(den,1e-6)).astype(np.float32)))
def ncc(a,b):
    a=a-a.mean(); b=b-b.mean()
    na=float(np.linalg.norm(a)); nb=float(np.linalg.norm(b))
    return 0.0 if na==0 or nb==0 else float((a*b).sum()/(na*nb))
def nccb(a,B):
    a=a-a.mean(); na=a.norm()
    Bc=B-B.mean(dim=(1,2),keepdim=True); nb=Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)

In [ ]:
# %% C0 — discover the 5 Kodak bodies, split, estimate fingerprints, LODO kappa
RX=re.compile(r"^(?P<model>.+)_(?P<inst>\d+)_(?P<shot>\d+)\.(jpe?g|JPE?G|png|PNG)$")
def c0():
    base=os.path.join(DRIVE,C.DRESDEN)
    dv={}
    for fp in glob.glob(os.path.join(base,"**","*.*"),recursive=True):
        m=RX.match(os.path.basename(fp))
        if not m or m["model"]!=C.CAM_MODEL: continue
        dv.setdefault(f"{m['model']}_{m['inst']}",[]).append(fp)
    for v in dv.values(): v.sort()
    need=C.N_E1+C.N_E2+C.N_T+C.N_H+3*C.GUARD
    print(f"[C0] {C.CAM_MODEL}: {len(dv)} bodies, need {need} images each")
    from collections import Counter
    picks={}
    for i,(did,files) in enumerate(sorted(dv.items(), key=lambda kv:-len(kv[1]))[:C.N_DEV]):
        cnt=Counter()
        for fp in files:
            try:
                with Image.open(fp) as im: cnt[im.size]+=1
            except Exception: pass
        size,_=cnt.most_common(1)[0]
        kept=[]
        for fp in files:
            try:
                with Image.open(fp) as im:
                    if im.size==size: kept.append(fp)
            except Exception: pass
        if min(size)<MEAS: HALT(f"{did}: {size} < {MEAS}")
        if len(kept)<need: HALT(f"{did}: {len(kept)} usable < {need}")
        picks[f"D{i}"]={"id":did,"size":list(size),"files":kept}
        print(f"   D{i} = {did:22s} {len(files):4d} imgs, {len(kept):4d} uniform @ {size}")
    if len(picks)<C.N_DEV: HALT(f"only {len(picks)} bodies with enough images")

    hdr=["dev","device_id","split","idx","path"]
    cp=os.path.join(ROOT,"manifest","manifest.csv")
    if os.path.exists(cp): os.remove(cp)
    man={"config_sha":CFG_SHA,"cam_model":C.CAM_MODEL,"picks":{},"meas":MEAS}
    for d,info in picks.items():
        k=info["files"]; g=C.GUARD; i=0; sp={}
        for nm,n in (("E1",C.N_E1),("E2",C.N_E2),("T",C.N_T),("H",C.N_H)):
            sp[nm]=k[i:i+n]; i+=n+g
        seen=set()
        for nm,v in sp.items():
            if seen&set(v): HALT(f"{d}: split overlap at {nm}")
            seen|=set(v)
            for j,fp in enumerate(v): append_row(cp,hdr,[d,info["id"],nm,j,fp])
        man["picks"][d]={"id":info["id"],"size":info["size"],
                         "n":{k_:len(v) for k_,v in sp.items()}}
    ajson(man,os.path.join(ROOT,"manifest","manifest.json"),indent=2)

    M={}
    for r in _csv.DictReader(open(cp)): M.setdefault((r["dev"],r["split"]),[]).append(r["path"])
    for d in DEVS():
        for which in ("E1","E2"):
            p=os.path.join(ROOT,"fingerprints",f"K_{d}_{which}.npy")
            if os.path.exists(p): continue
            t0=time.time(); np.save(p,estK(M[(d,which)]))
            print(f"[C0] K_{d}_{which} from {len(M[(d,which)])} imgs ({time.time()-t0:.0f}s)")
    K2={d:np.load(os.path.join(ROOT,"fingerprints",f"K_{d}_E2.npy")) for d in DEVS()}
    # leave-one-device-out camera-model component
    for d in DEVS():
        kap=np.mean([K2[j] for j in DEVS() if j!=d],axis=0).astype(np.float32)
        res=(K2[d]-kap*(ncc(K2[d],kap)*np.linalg.norm(K2[d])/max(np.linalg.norm(kap),1e-12))
             ).astype(np.float32)
        np.save(os.path.join(ROOT,"fingerprints",f"K_{d}_E2_lodo.npy"),res)
    print("\n[C0] pairwise NCC between device fingerprints (the kappa_model floor):")
    for a,b in itertools.combinations(DEVS(),2):
        print(f"   {a}-{b}: raw {ncc(K2[a],K2[b]):+.4f}   lodo "
              f"{ncc(np.load(os.path.join(ROOT,'fingerprints',f'K_{a}_E2_lodo.npy')),
                     np.load(os.path.join(ROOT,'fingerprints',f'K_{b}_E2_lodo.npy'))):+.4f}")
    # positive control on H
    hp=os.path.join(ROOT,"csv","c0_positive_control.csv")
    if not os.path.exists(hp):
        for d in DEVS():
            for i,fp in enumerate(M[(d,"H")]):
                Y=load_lum(fp); W=wres(Y)
                for k_ in DEVS():
                    append_row(hp,["img_dev","K_dev","idx","rho"],[d,k_,i,f"{ncc(W,Y*K2[k_]):.6e}"])
            print(f"[C0] positive control {d} done")
    pc=pd.read_csv(hp)
    same=pc[pc.img_dev==pc.K_dev]["rho"].to_numpy(); cross=pc[pc.img_dev!=pc.K_dev]["rho"].to_numpy()
    lab=np.r_[np.ones(len(same)),np.zeros(len(cross))]; sc=np.r_[same,cross]
    o=np.argsort(sc); rk=np.empty(len(sc)); rk[o]=np.arange(len(sc))
    auc=float((rk[lab==1].mean()-(lab.sum()-1)/2)/(len(sc)-lab.sum()))
    print(f"\n[C0] same-vs-cross device AUC across 5 bodies = {auc:.4f}")
    if auc<0.99: HALT(f"K-quality gate: AUC {auc:.4f} < 0.99")
    ajson({"auc":auc,"config_sha":CFG_SHA},os.path.join(ROOT,"fingerprints","c0_gates.json"),indent=2)
    print("[C0] PASS")
if stage("C0"): c0()

In [ ]:
# %% C1 — materialise training PNGs, then train 10 adapters (5 devices x 2 seeds)
def MANIF():
    M={}
    for r in _csv.DictReader(open(os.path.join(ROOT,"manifest","manifest.csv"))):
        M.setdefault((r["dev"],r["split"]),[]).append(r["path"])
    return M
def tdir(d): return os.path.join(ROOT,"train_png",d)
def adir(t): return os.path.join(ROOT,"adapters",t)
def _exp(t):
    _,d,s=next(a for a in all_arms() if a[0]==t)
    return {"tag":t,"dev":d,"seed":s,"rank":C.RANK,"steps":C.STEPS,"meas":MEAS,
            "config_sha":CFG_SHA,"model":C.MODEL,"dtype":str(DT)}
def avalid(t):
    p=os.path.join(adir(t),"train_meta.json")
    if not os.path.exists(os.path.join(adir(t),"pytorch_lora_weights.safetensors")): return False
    m=json.load(open(p)) if os.path.exists(p) else None
    return m is not None and all(m.get(k)==v for k,v in _exp(t).items()) and m.get("effective",False)

def c1():
    M=MANIF()
    for d in DEVS():
        o=tdir(d); os.makedirs(o,exist_ok=True)
        if len(glob.glob(os.path.join(o,"*.png")))>=C.N_T: continue
        for i,fp in enumerate(M[(d,"T")]):
            Image.fromarray(load_rgb(fp).astype(np.uint8)).save(
                os.path.join(o,f"{i:04d}.png"),compress_level=1)
        print(f"[C1] materialised {d}: {C.N_T} PNGs")
    from diffusers import StableDiffusion3Pipeline
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    pe_p=os.path.join(ROOT,"adapters","_prompt.pt")
    if os.path.exists(pe_p):
        z=torch.load(pe_p,map_location=DEV); pe,pp=z["pe"].to(DEV),z["pp"].to(DEV)
    else:
        pi=StableDiffusion3Pipeline.from_pretrained(C.MODEL,torch_dtype=DT,token=True).to(DEV)
        with torch.no_grad():
            pe,_,pp,_=pi.encode_prompt(prompt=C.CAPTION,prompt_2=C.CAPTION,prompt_3=C.CAPTION,
                                       device=DEV,num_images_per_prompt=1,
                                       do_classifier_free_guidance=False)
        torch.save({"pe":pe.cpu(),"pp":pp.cpu()},pe_p); del pi
        gc.collect(); torch.cuda.empty_cache()
    for n,(tag,d,seed) in enumerate(all_arms(),1):
        if avalid(tag): print(f"[C1] {n}/{len(all_arms())} skip {tag}"); continue
        out=adir(tag)
        if os.path.exists(out): shutil.move(out,out+f"__stale_{int(time.time())}")
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        try:
            pipe=StableDiffusion3Pipeline.from_pretrained(
                C.MODEL,torch_dtype=DT,token=True,text_encoder=None,text_encoder_2=None,
                text_encoder_3=None,tokenizer=None,tokenizer_2=None,tokenizer_3=None).to(DEV)
        except Exception:
            pipe=StableDiffusion3Pipeline.from_pretrained(C.MODEL,torch_dtype=DT,token=True).to(DEV)
            pipe.text_encoder=pipe.text_encoder_2=pipe.text_encoder_3=None
        sch,vae,tr=pipe.scheduler,pipe.vae,pipe.transformer
        vae.requires_grad_(False); tr.requires_grad_(False)
        tr.add_adapter(LoraConfig(r=C.RANK,lora_alpha=C.RANK,init_lora_weights="gaussian",
                                  target_modules=list(C.TARGETS)))
        if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
        tr.train()
        ps=[p for p in tr.parameters() if p.requires_grad]
        opt=torch.optim.AdamW(ps,lr=C.LR,weight_decay=1e-4)
        lrs=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(C.STEPS/C.GRAD_ACC))
        opt.zero_grad(set_to_none=True)
        files=sorted(glob.glob(os.path.join(tdir(d),"*.png")))
        cache=torch.stack([torch.from_numpy(
            np.asarray(Image.open(f).convert("RGB"),np.float32).copy()/127.5-1.0).permute(2,0,1)
            for f in files])
        sf=float(vae.config.scaling_factor); sh=float(vae.config.shift_factor or 0.0)
        ts,sg=sch.timesteps.to(DEV),sch.sigmas.to(DEV); nts=int(sch.config.num_train_timesteps)
        if len(sg)!=nts or len(ts)!=nts: HALT("scheduler array length mismatch — pin diffusers")
        g=torch.Generator().manual_seed(seed); losses=[]; t0=time.time()
        print(f"\n[C1] {n}/{len(all_arms())} {tag}: dev={d} seed={seed}")
        for st in range(C.STEPS):
            idx=torch.randint(0,cache.shape[0],(C.BATCH,),generator=g)
            px=cache[idx].to(DEV,dtype=DT)
            with torch.no_grad():
                lat=vae.encode(px).latent_dist.sample(); lat=((lat-sh)*sf).to(DT)
            noise=torch.randn_like(lat)
            u=torch.sigmoid(torch.randn(lat.shape[0],device=DEV,dtype=torch.float32))
            ii=(u*nts).long().clamp_(0,nts-1); sig=sg[ii].to(lat.dtype).view(-1,1,1,1)
            pred=tr(hidden_states=((1.0-sig)*lat+sig*noise).to(DT),timestep=ts[ii],
                    encoder_hidden_states=pe.repeat(lat.shape[0],1,1).to(DT),
                    pooled_projections=pp.repeat(lat.shape[0],1).to(DT),return_dict=False)[0]
            loss=tF.mse_loss(pred.float(),(noise-lat).float())/C.GRAD_ACC
            loss.backward()
            if (st+1)%C.GRAD_ACC==0:
                torch.nn.utils.clip_grad_norm_(ps,1.0); opt.step(); lrs.step()
                opt.zero_grad(set_to_none=True)
            losses.append(float(loss.detach())*C.GRAD_ACC)
            if not np.isfinite(losses[-1]): HALT(f"{tag}: non-finite loss at step {st+1}")
            if (st+1)%400==0:
                el=time.time()-t0
                print(f"    {st+1}/{C.STEPS} loss {np.mean(losses[-400:]):.5f} "
                      f"eta {(el/(st+1))*(C.STEPS-st-1)/60:.1f}min")
        sd=get_peft_model_state_dict(tr)
        bn=float(sum(float(v.float().norm()) for k,v in sd.items() if "lora_B" in k))
        if not (bn>1e-6 and all(bool(torch.isfinite(v).all()) for v in sd.values())):
            HALT(f"{tag}: DEAD adapter (||lora_B||={bn:.3e})")
        os.makedirs(out,exist_ok=True)
        StableDiffusion3Pipeline.save_lora_weights(save_directory=out,transformer_lora_layers=sd)
        m=_exp(tag); m.update({"loss_tail":float(np.mean(losses[-100:])),"lora_B_norm":bn,
                               "effective":True,"minutes":(time.time()-t0)/60})
        ajson(m,os.path.join(out,"train_meta.json"),indent=2)
        print(f"[C1] saved {tag}: ||lora_B||={bn:.2f} tail={m['loss_tail']:.5f} "
              f"({m['minutes']:.1f} min)")
        del pipe,tr,vae,cache; gc.collect(); torch.cuda.empty_cache()
    bad=[t for t in all_tags() if not avalid(t)]
    if bad: HALT(f"C1 incomplete: {bad}")
    print(f"[C1] PASS — {len(all_tags())} adapters")
if stage("C1"): c1()

In [ ]:
# %% C2 — generation, paired seed bank shared with the base study
def gdir(t): return os.path.join(ROOT,"gens",t)
def c2():
    if DEV!="cuda": HALT("C2 needs a GPU")
    from diffusers import StableDiffusion3Pipeline
    bank=[(C.CAPTION,C.GEN_SEED_BASE+j) for j in range(C.G_PER)]
    pipe=StableDiffusion3Pipeline.from_pretrained(C.MODEL,torch_dtype=DT,token=True).to(DEV)
    pipe.set_progress_bar_config(disable=True); prev=False
    try:
        for tag in all_tags():
            d_=gdir(tag); os.makedirs(d_,exist_ok=True)
            for t_ in glob.glob(os.path.join(d_,"*.tmp")): os.remove(t_)
            miss=[i for i in range(C.G_PER)
                  if not os.path.exists(os.path.join(d_,f"{i:05d}.png"))]
            if not miss: print(f"[C2] skip {tag}"); continue
            if prev: pipe.unload_lora_weights(); prev=False
            pipe.load_lora_weights(adir(tag)); prev=True
            print(f"[C2] {tag}: {len(miss)} missing")
            for s in range(0,len(miss),C.GEN_BATCH):
                ix=miss[s:s+C.GEN_BATCH]
                gs=[torch.Generator(device=DEV).manual_seed(bank[i][1]) for i in ix]
                with torch.inference_mode():
                    imgs=pipe(prompt=[bank[i][0] for i in ix],num_inference_steps=C.GEN_STEPS,
                              guidance_scale=C.CFG_SCALE,height=MEAS,width=MEAS,generator=gs).images
                for i,im in zip(ix,imgs):
                    t_=os.path.join(d_,f"{i:05d}.png.tmp")
                    im.save(t_,format="PNG"); os.replace(t_,os.path.join(d_,f"{i:05d}.png"))
                if (s//C.GEN_BATCH)%25==0: print(f"    {s+len(ix)}/{len(miss)}")
            print(f"[C2] complete {tag}")
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    print("[C2] PASS")
if stage("C2"): c2()

In [ ]:
# %% C3 — measurement against all 5 device fingerprints, raw and LODO-residualised
def c3():
    for variant in ("raw","lodo"):
        suf="" if variant=="raw" else "_lodo"
        K=torch.stack([torch.from_numpy(
            np.load(os.path.join(ROOT,"fingerprints",f"K_{d}_E2{suf}.npy"))) for d in DEVS()]).to(_MD)
        p=os.path.join(ROOT,"csv",f"c3_measure_{variant}.csv")
        done=done_keys(p,["tag","gen_idx","K"])
        ex=ThreadPoolExecutor(max_workers=4)
        try:
            for tag in all_tags():
                paths=[os.path.join(gdir(tag),f"{i:05d}.png") for i in range(C.G_PER)]
                todo=[i for i in range(C.G_PER)
                      if not all((tag,str(i),d) in done for d in DEVS())]
                if not todo: continue
                fut={i:ex.submit(load_lum,paths[i],False) for i in todo[:6]}
                t0=time.time()
                for pos,i in enumerate(todo):
                    Z_np=fut.pop(i).result(); nx=pos+6
                    if nx<len(todo): fut[todo[nx]]=ex.submit(load_lum,paths[todo[nx]],False)
                    W=torch.from_numpy(wres(Z_np)).to(_MD); Z=torch.from_numpy(Z_np).to(_MD)
                    v=nccb(W,Z.unsqueeze(0)*K).cpu().numpy()
                    for j,d in enumerate(DEVS()):
                        if (tag,str(i),d) in done: continue
                        append_row(p,["tag","gen_idx","K","rho"],[tag,i,d,f"{v[j]:.6e}"])
                print(f"[C3] {variant} {tag}: {len(todo)} ({time.time()-t0:.0f}s)")
        finally:
            ex.shutdown(wait=True); gc.collect()
            if DEV=="cuda": torch.cuda.empty_cache()
        n=len(done_keys(p,["tag","gen_idx","K"])); exp=len(all_tags())*C.G_PER*len(DEVS())
        print(f"[C3] {variant}: {n}/{exp} rows")
        if n!=exp: HALT(f"C3 {variant} incomplete")
    print("[C3] PASS")
if stage("C3"): c3()

In [ ]:
# %% C4 — mixed model: separate between-device from adapter-training variance
def c4():
    rng=np.random.default_rng(0)
    out={"config_sha":CFG_SHA,"variants":{}}
    for variant in ("raw","lodo"):
        d=pd.read_csv(os.path.join(ROOT,"csv",f"c3_measure_{variant}.csv"))
        rows=[]
        for tag,dev,seed in all_arms():
            g=d[d.tag==tag]
            own=g[g.K==dev].sort_values("gen_idx")["rho"].to_numpy(float)
            oth=[g[g.K==o].sort_values("gen_idx")["rho"].to_numpy(float) for o in DEVS() if o!=dev]
            n=min([len(own)]+[len(x) for x in oth])
            th=own[:n]-np.mean([x[:n] for x in oth],axis=0)   # own vs the mean of the other four
            m=json.load(open(os.path.join(adir(tag),"train_meta.json")))
            rows.append({"tag":tag,"dev":dev,"seed":seed,"theta":float(th.mean()),
                         "se_img":float(th.std(ddof=1)/np.sqrt(len(th))),
                         "lora_B":m.get("lora_B_norm"),"tail":m.get("loss_tail")})
        R=pd.DataFrame(rows)
        print(f"\n=== {variant.upper()} ===")
        print(R.to_string(index=False,float_format=lambda x:f"{x:.5g}"))
        # variance decomposition: theta_ds = mu + u_d + v_ds
        dm=R.groupby("dev")["theta"].mean()
        gm=float(R.theta.mean()); k=len(dm); nps=len(C.SEEDS)
        ms_dev=float(nps*((dm-gm)**2).sum()/(k-1))
        within=float(sum(((R[R.dev==dv].theta-dm[dv])**2).sum() for dv in dm)/(k*(nps-1)))
        var_u=max((ms_dev-within)/nps,0.0)
        se_dev=float(dm.std(ddof=1)/np.sqrt(k))
        tcrit=float(sps.t.ppf(1-C.ALPHA/2,df=k-1))
        U=float(gm+tcrit*se_dev)
        # device-level sign-flip on the 5 device means
        obs=dm.mean(); cnt=0
        for _ in range(C.N_PERM):
            cnt+=((dm.to_numpy()*rng.choice((-1.,1.),k)).mean()>=obs)
        p=(cnt+1)/(C.N_PERM+1)
        out["variants"][variant]={"per_adapter":rows,"device_means":dm.to_dict(),
            "grand_mean":gm,"var_between_device":var_u,"var_within_device":within,
            "icc":float(var_u/(var_u+within)) if (var_u+within)>0 else 0.0,
            "t_crit":tcrit,"U_device_level":U,"p_signflip_devices":float(p),"n_devices":k}
        print(f"\n  grand mean theta      = {gm:+.4e}")
        print(f"  between-device var u  = {var_u:.4e}   within-device (seed) var = {within:.4e}")
        print(f"  ICC (device share)    = {out['variants'][variant]['icc']:.3f}")
        print(f"  device-level U (99%)  = {U:+.4e}   sign-flip p = {p:.4g}")
    ajson(out,os.path.join(ROOT,"C4_multidev.json"),indent=2)
    print("\nREAD: ICC near 0 means adapter-training variance dominates and device identity")
    print("      contributes little — consistent with the base-study null. Report BOTH the raw")
    print("      and the LODO-residualised analysis; LODO is what five bodies buy you.")
if stage("C4"): c4()